In [1]:
import warnings
warnings.simplefilter('ignore')

In [2]:
import os, sys
from typing import Optional

import polars as pl
import pandas as pd

REPO_DATASET_PATH = "/kaggle/input/olympiadlevelmaths4llm"
sys.path.append(REPO_DATASET_PATH + "/src")

# --- Runtime / platform guards ---
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TRANSFORMERS_NO_FLAX"] = "1"
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"
os.environ["TRITON_PTXAS_PATH"] = "/usr/local/cuda/bin/ptxas"
os.environ["TIKTOKEN_ENCODINGS_BASE"] = (
    "/kaggle/usr/lib/aimo3_packages_offline/tiktoken_encodings"
)

# =============================================================================
# V2 CONFIG — only env vars that AIMO3Config.from_env() actually reads
# Keep prompts compact so they do not overshadow short/easy problems.
# =============================================================================

# --- Prompts (override defaults from config.py) ---
os.environ["AIMO3_SYSTEM_PROMPT"] = (
    "Solve for the correct integer answer. "
    "Find the fastest exact method first. "
    "Prefer reductions, formulas, invariants, modular arithmetic, and efficient counting over long proofs. "
    "Use Python early for small cases, pattern checks, and the final exact computation. "
    "Avoid expensive brute force unless it is clearly tiny. "
    "Return only the final verified integer in \\boxed{n}, where n is in [0, 99999]."
)

os.environ["AIMO3_TOOL_PROMPT"] = (
    "Use the Python notebook to find and verify the fastest exact computation. "
    "Start with small cases or symbolic simplification, then implement the best exact algorithm. "
    "Check complexity before loops, keep code short, and print only decisive results. "
    "Print VERIFY_OK only when the computation supports the candidate answer. "
    "Always use print(). Default timeout is 30s; for heavier code add '# timeout: 120' on the first line."
)

os.environ["AIMO3_PREFERENCE_PROMPT"] = (
    "Find the most efficient exact method first. Use Python for small cases and the final exact computation. "
    "Avoid long prose and avoid brute force unless it is clearly tiny. Final answer: \\boxed{n}."
)

# --- Model / server ---
os.environ["AIMO3_MODEL_PATH"] = "/kaggle/input/gpt-oss-120b/transformers/default/1"
os.environ["AIMO3_SERVED_MODEL_NAME"] = "gpt-oss"
os.environ["AIMO3_REUSE_EXISTING_SERVER"] = "1"
os.environ["AIMO3_SERVER_TIMEOUT"] = "1500"
os.environ["AIMO3_REQUIRE_CUDA"] = "1"

# Cold-start speedup
os.environ["AIMO3_PRELOAD_MODEL_WEIGHTS"] = "1"
os.environ["AIMO3_PRELOAD_MODEL_WORKERS"] = "8"

# --- Display / tracing ---
os.environ["AIMO3_DISPLAY_CANDIDATES"] = "1"
os.environ["AIMO3_TRACE"] = "1"
os.environ["AIMO3_TRACE_ATTEMPTS"] = "0"
os.environ["AIMO3_TRACE_ENV"] = "1"
os.environ["AIMO3_TRACE_ENV_PACKAGES"] = "sympy,numpy,mpmath,jupyter_client,ortools,z3-solver"
os.environ["AIMO3_TRACE_INCLUDE_PROBLEM_TEXT"] = "0"

# --- Core decoding / capacity ---
os.environ["AIMO3_SEED"] = "42"
os.environ["AIMO3_SEARCH_TOKENS"] = "32"
os.environ["AIMO3_CONTEXT_TOKENS"] = "65536"
os.environ["AIMO3_BATCH_SIZE"] = "128"
os.environ["AIMO3_GPU_MEMORY_UTILIZATION"] = "0.96"

# --- Sandbox/tooling ---
os.environ["AIMO3_JUPYTER_TIMEOUT"] = "30"
os.environ["AIMO3_SANDBOX_TIMEOUT"] = "2"

# --- Time budgeting ---
os.environ["AIMO3_PROBLEMS_TOTAL"] = "50"
os.environ["AIMO3_NOTEBOOK_LIMIT"] = "17700"
os.environ["AIMO3_BASE_PROBLEM_TIMEOUT"] = "280"
os.environ["AIMO3_HIGH_PROBLEM_TIMEOUT"] = "1500"

# --- Attempt scheduling ---
os.environ["AIMO3_ATTEMPTS"] = "5"
os.environ["AIMO3_WORKERS"] = "8"
os.environ["AIMO3_TURNS"] = "128"
os.environ["AIMO3_EARLY_STOP"] = "4"
os.environ["AIMO3_EARLY_STOP_MIN_VERIFIED"] = "0"

# --- Extraction ---
os.environ["AIMO3_STRICT_FALLBACK_EXTRACTION"] = "1"

# --- Decoding knobs ---
os.environ["AIMO3_TEMPERATURE"] = "1.0"
os.environ["AIMO3_MIN_P"] = "0.02"
os.environ["AIMO3_TOP_P"] = "0.98"
os.environ["AIMO3_TOP_K"] = "-1"

os.environ["AIMO3_VERIFY_PHASE_ENABLED"] = "0"
os.environ["AIMO3_VERIFY_DISABLE_GLOBALLY_IF_ALL_UNKNOWN"] = "1"
os.environ["AIMO3_VERIFY_ATTEMPTS_PER_CANDIDATE"] = "2"
os.environ["AIMO3_VERIFY_TOP_K_CANDIDATES"] = "2"
os.environ["AIMO3_VERIFY_TIMEOUT"] = "30"
os.environ["AIMO3_VERIFY_TEMPERATURE"] = "0.3"
os.environ["AIMO3_PYTHON_TOOL_VERIFY_REQUIRE_MARKER"] = "0"

# --- Time Management Approach ---
# - 'equal': use equal share of remaining time
# - 'base': use configured base_timeout_s for every problem
# - 'avg': use rolling average
# - 'cumulative': add carryover from previous problems
# - 'hybrid': take max(equal, avg, base) (default behavior)
os.environ["AIMO3_BUDGET_STRATEGY"] = "cumulative"
os.environ["AIMO3_BASE_TIMEOUT_S"] = ""  # unset to avoid overriding base_problem_timeout
os.environ["AIMO3_CARRYOVER_ENABLED"] = "1"
os.environ["AIMO3_CUMULATIVE_DISTRIBUTE"] = "0"

os.environ["AIMO3_WICKELGREN"] = "1"
os.environ["AIMO3_TRACE_ATTEMPTS"] = "1"
os.environ["AIMO3_TRACE_ATTEMPTS_MAX_CHARS"] = "60000"

os.environ["AIMO3_FILTER_TO_VERIFIED_IF_ANY"] = "0"
os.environ["AIMO3_ENTROPY_WEIGHTING"] = "0"
os.environ["AIMO3_RANKING_STRATEGY"] = "votes_then_verified"

# --- CPU retriever (v2, v1-compatible env names) ---
# Set this path to your mounted KB dir containing concepts.json or concepts.pkl
os.environ["AIMO3_RETRIEVER_ENABLED"] = "0"
os.environ["AIMO3_RETRIEVER_KB_PATH"] = "/kaggle/input/olympiadlevelmaths4llmdb/OlympiadLevelMaths4LLMDB/knowledge_base"
os.environ["AIMO3_RETRIEVER_CPU_ONLY"] = "1"
os.environ["AIMO3_RETRIEVER_TOP_K"] = "5"
os.environ["AIMO3_RETRIEVER_MIN_SCORE"] = "0.08"
os.environ["AIMO3_RETRIEVER_INCLUDE_EXAMPLES"] = "1"
os.environ["AIMO3_RETRIEVER_INCLUDE_DEFINITIONS"] = "1"
os.environ["AIMO3_RETRIEVER_WARMUP_ON_INIT"] = "1"
os.environ["AIMO3_RETRIEVER_MODEL_PATH"] = "/kaggle/input/models/srg9000/all-minilm-l6-v2/transformers/default/1/all-MiniLM-L6-v2"

os.environ["AIMO3_ADAPTIVE_BUDGET_FLEX_POOL_FRACTION"] = "0"

# --- Meta Learning ---
os.environ["AIMO3_META_LEARNING_ENABLED"] = "0"
os.environ["AIMO3_META_LEARNING_SIMILARITY_THRESHOLD"] = "0.3"

os.environ["AIMO3_Z3_TOOL_ENABLED"] = "0"


os.environ["AIMO3_50_PROBLEMS_DATA_ENABLED"] = "1"

# Misc
os.environ["PYDEVD_DISABLE_FILE_VALIDATION"] = "1"

MODEL_PATH = os.getenv("AIMO3_MODEL_PATH", "")
print(
    f"Config: BASE_TIMEOUT={os.environ.get('AIMO3_BASE_PROBLEM_TIMEOUT')}s, HIGH_TIMEOUT={os.environ.get('AIMO3_HIGH_PROBLEM_TIMEOUT')}s"
)
print(
    f"Config: EARLY_STOP={os.environ.get('AIMO3_EARLY_STOP')}, MIN_VERIFIED={os.environ.get('AIMO3_EARLY_STOP_MIN_VERIFIED')}"
)
print(
    f"Config: ATTEMPTS={os.environ.get('AIMO3_ATTEMPTS')}, WORKERS={os.environ.get('AIMO3_WORKERS')}, TURNS={os.environ.get('AIMO3_TURNS')}"
)
print(
    f"Config: RANKING_STRATEGY={os.environ.get('AIMO3_RANKING_STRATEGY')}, FILTER_TO_VERIFIED_IF_ANY={os.environ.get('AIMO3_FILTER_TO_VERIFIED_IF_ANY')}"
)
print(
    f"Config: RETRIEVER_ENABLED={os.environ.get('AIMO3_RETRIEVER_ENABLED')}, RETRIEVER_KB_PATH={os.environ.get('AIMO3_RETRIEVER_KB_PATH')}"
)
MODEL_PATH

Config: BASE_TIMEOUT=280s, HIGH_TIMEOUT=1500s
Config: EARLY_STOP=4, MIN_VERIFIED=0
Config: ATTEMPTS=5, WORKERS=8, TURNS=128
Config: RANKING_STRATEGY=votes_then_verified, FILTER_TO_VERIFIED_IF_ANY=0
Config: RETRIEVER_ENABLED=0, RETRIEVER_KB_PATH=/kaggle/input/olympiadlevelmaths4llmdb/OlympiadLevelMaths4LLMDB/knowledge_base


'/kaggle/input/gpt-oss-120b/transformers/default/1'

In [3]:
# --- First-wave fast-answer mode (helps capture easy consensus before full reasoning) ---
os.environ["AIMO3_ANSWER_ONLY_ATTEMPTS"] = "0"
os.environ["AIMO3_ANSWER_ONLY_PROMPT"] = (
    "Find the integer with the fastest exact method. "
    "Use efficient reasoning, prefer a short exact algorithm, and do not overcomplicate easy problems. "
    "Output only the final verified integer in \\boxed{number}."
 )

# Re-enable entropy-weighted tie-breaking to recover a confidence signal across attempts.
os.environ["AIMO3_ENTROPY_WEIGHTING"] = "1"

print(
    f"Config: ANSWER_ONLY_ATTEMPTS={os.environ.get('AIMO3_ANSWER_ONLY_ATTEMPTS')}, "
    f"ENTROPY_WEIGHTING={os.environ.get('AIMO3_ENTROPY_WEIGHTING')}"
 )

Config: ANSWER_ONLY_ATTEMPTS=0, ENTROPY_WEIGHTING=1


In [4]:
from olympiad_llm.aimo3.v2.cleanup import environ_setup_parallel, wait_all

# Start environment setup (uninstall + offline install + model warmup) ALL IN PARALLEL.
# This overlaps pip install with model cache warmup, saving ~30-60s on cold start.
ENVIRON_SETUP = environ_setup_parallel(warm_model=True, model_workers=8)
print("Started parallel environment setup:")
print("  - pip uninstall conflicts (background)")
print("  - pip install required packages (background)")
print("  - Model weight cache warmup (background)")
print("Will wait for completion lazily when the solver is first needed.")

Started parallel environment setup:
  - pip uninstall conflicts (background)
  - pip install required packages (background)
  - Model weight cache warmup (background)
Will wait for completion lazily when the solver is first needed.


In [5]:
import threading
from olympiad_llm.aimo3.v2.config import AIMO3Config
from olympiad_llm.aimo3.v2.runner import build_solver, run_kaggle_inference
from olympiad_llm.aimo3.v2.cleanup import wait_all

# IMPORTANT for Kaggle: don't block notebook execution on vLLM cold-start here.
# We'll build the solver lazily on the first predict() call.
cfg = AIMO3Config.from_env()
solver = None
_solver_lock = threading.Lock()

def get_solver():
    global solver
    if solver is not None:
        return solver
    with _solver_lock:
        if solver is not None:
            return solver
        # Wait for ALL parallel setup tasks to finish (pip + model warmup).
        if "ENVIRON_SETUP" in globals() and isinstance(ENVIRON_SETUP, dict):
            wait_all(ENVIRON_SETUP, timeout=180)
            print("✓ Parallel setup complete (pip + model cache warmup)")
        solver = build_solver(cfg)
        return solver

print("Lazy solver configured. Inference server can start now.")

Lazy solver configured. Inference server can start now.


In [6]:
def predict(id_: pl.DataFrame, question: pl.DataFrame, answer: Optional[pl.DataFrame] = None) -> pl.DataFrame:
    global correct_count, total_count, predictions, ground_truth
    
    question_id = id_.item(0)
    question_text = question.item(0)
    
    print("------")
    print(f"ID: {question_id}")
    
    # Build solver only when needed (keeps Kaggle inference server startup fast).
    s = get_solver()
    final_answer = s.solve_problem(question_text)
    predictions[question_id] = final_answer

    # Check accuracy if ground truth available (local runs only).
    total_count += 1
    if question_id in ground_truth:
        gt = ground_truth[question_id]
        is_correct = (final_answer == gt)
        if is_correct:
            correct_count += 1
        status = "✅" if is_correct else "❌"
        print(f"Answer: {final_answer} | Ground Truth: {gt} | {status}")
        print(f"📊 Running Accuracy: {correct_count}/{total_count} ({100*correct_count/total_count:.1f}%)")
    else:
        print(f"Answer: {final_answer}")
    
    print("------\n")
    
    return pl.DataFrame({'id': question_id, 'answer': final_answer})

In [7]:
from olympiad_llm.aimo3.prepare import prepare_reference_csv

In [8]:
# Load ground truth only for local testing (avoid delaying server start in competition reruns).
if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    ground_truth = {}
elif int(os.environ.get('AIMO3_50_PROBLEMS_DATA_ENABLED')):
    df = pd.read_csv('/kaggle/input/datasets/amanatar/50problems/50problems.csv')
    df.insert(0, 'id', range(1, len(df) + 1))
    df.rename(columns={'Problem': 'problem', 'Answer': 'answer'}, inplace=True)
    print(df.head())
    df.to_csv('50problems.csv', index=False)
    ground_truth, _ = prepare_reference_csv(
        "50problems.csv",
    )
else:
    ground_truth, _ = prepare_reference_csv(
        "/kaggle/input/ai-mathematical-olympiad-progress-prize-3/reference.csv",
        # "/kaggle/input/olympiadlevelmaths4llmdb/inmo_1986.csv",
        # problem_ids=["dd7f5e", "86e8e5"],
        # problem_ids=["86e8e5"],
    )

# Track predictions for accuracy calculation
predictions = {}
correct_count = 0
total_count = 0

   id                                            problem  answer
0   1  Let $ABC$ be an acute-angled triangle with int...     336
1   2  Define a function $f \colon \mathbb{Z}_{\geq 1...   32951
2   3  A tournament is held with $2^{20}$ runners eac...   21818
3   4  Ken writes a positive integer $n$ on a blackbo...   32193
4   5  Let triangle $ABC$ be $n$-tastic if $BD = F_n,...   57447


In [9]:
inference_server = run_kaggle_inference(predict)

if os.getenv('KAGGLE_IS_COMPETITION_RERUN'):
    inference_server.serve()
else:
    inference_server.run_local_gateway(("reference.csv",))

------
ID: 28
✓ Parallel setup complete (pip + model cache warmup)

Problem: Let $b \geq 2$. Call a positive integer $b$-eautiful if it has exactly two digits in base $b$ that sum to $\sqrt{n}$. Find the least integer $b$ for which there are more than ten $b$-eautiful integers.

Budget: 354.00s | [Budget] 0/50 done | Remaining: 17700s | Flex: 0s/0s | Avg: 280s | Next: 354s | Extensions: 0



,Attempt,Answer,ToolVerified,PyCalls,Timeouts,PyErrors,Tokens,Entropy,Snippet,LastError
0,1,211,True,7,0,0,4360,0.730677,...provide the method summarised: derived equa...,None
1,2,211,True,6,0,0,6156,0.720578,...erpretation is right.\n\nThus answer is 211...,None
2,4,211,True,6,0,0,5248,0.667664,...nt.assistantanalysis to=python codeprint(co...,None
3,5,211,True,9,0,0,4253,0.682238,... has exactly two digits in base b.\n\nThus ...,None



Final Answer: 211 (votes=4, verified=4)

Answer: 211 | Ground Truth: 211 | ✅
📊 Running Accuracy: 1/1 (100.0%)
------

------
ID: 36

Problem: Call a positive integer extra-distinct if remainders when divided by 2, 3, 4, 5, and 6 are distinct. Find the number of extra-distinct positive integers less than 1000.

Budget: 640.76s | [Budget] 1/50 done | Remaining: 17633s | Flex: 0s/0s | Avg: 67s | Next: 360s | Extensions: 0



,Attempt,Answer,ToolVerified,PyCalls,Timeouts,PyErrors,Tokens,Entropy,Snippet,LastError
0,1,49,True,5,0,0,1605,0.673338,"... distinct.\n\nCheck n=58: 58%2=0, %3=1, %4=...",None
1,4,49,True,4,0,0,1644,0.661449,... \boxed{49}.\n\nBut we might need to provid...,None
2,5,49,True,6,0,0,1569,0.712686,...+1=49. This matches. So answer 49.\n\nThus ...,None



Final Answer: 49 (votes=3, verified=3)

Answer: 49 | Ground Truth: 49 | ✅
📊 Running Accuracy: 2/2 (100.0%)
------

------
ID: 44

Problem: Circles $\omega_1, \omega_2$ intersect at $P, Q$. Parallel line $AB$ through $P$ forms trapezoid $XABY$. If $PX=10, PY=14, PQ=5$, find $m+n$ if the area is $m\sqrt{n}$.

Budget: 978.90s | [Budget] 2/50 done | Remaining: 17617s | Flex: 0s/0s | Avg: 42s | Next: 367s | Extensions: 0



...


Final Answer: 336 (votes=4, verified=4)

Answer: 336 | Ground Truth: 336 | ✅
📊 Running Accuracy: 40/48 (83.3%)
------

------
ID: 17

Problem: Eight circles of radius 34 can be placed tangent to $BC$ of $\triangle ABC$ sequentially tangent to each other, first to $AB$ and last to $AC$. Similarly, 2024 circles of radius 1 can be placed the same way. Find $m+n$ if the inradius is $m/n$.

Budget: 1500.00s | [Budget] 48/50 done | Remaining: 12066s | Flex: 0s/0s | Avg: 87s | Next: 1500s | Extensions: 0



,Attempt,Answer,ToolVerified,PyCalls,Timeouts,PyErrors,Tokens,Entropy,Snippet,LastError
0,1,197,True,3,0,0,10441,0.649519,...hus answer is 197.\n\nWe should ensure no m...,None
1,3,197,True,1,0,0,12173,0.764270,"...t \boxed{197}.\n\nBut wait, they ask ""Find ...",None
2,4,197,True,4,0,0,10283,0.759734,... - r)/ρ ). So (N-1)*2r = L*(ρ - r)/ρ => L =...,None
3,5,197,True,3,0,0,8803,0.700582,"...\nden = 1190\ng = math.gcd(num, den)\nnum//...",None



Final Answer: 197 (votes=4, verified=4)

Answer: 197 | Ground Truth: 540 | ❌
📊 Running Accuracy: 40/49 (81.6%)
------

------
ID: 8

Problem: Let $f \colon \mathbb{Z}_{\geq 1} \to \mathbb{Z}_{\geq 1}$ satisfy $f(m) + f(n) = f(m + n + mn)$ for all $m, n$. Across all functions where $f(n) \leq 1000$ for all $n \leq 1000$, how many different values can $f(2024)$ take?

Budget: 1500.00s | [Budget] 49/50 done | Remaining: 11965s | Flex: 0s/0s | Avg: 89s | Next: 1500s | Extensions: 0



,Attempt,Answer,ToolVerified,PyCalls,Timeouts,PyErrors,Tokens,Entropy,Snippet,LastError
0,1,580,False,6,0,1,11272,0.752210,...e increments are 2 (change a by 1 changes b...,----------------------------------------------...
1,2,580,True,1,0,0,7297,0.827051,...f possible values for 4a+2b = 2(2a+b) as ev...,None
2,4,580,True,10,0,0,11596,0.733912,... all functions where f(n) ≤ 1000 for all n ...,None
3,5,580,True,5,0,0,14971,0.744209,"... if check_pair(a3,a5):\n feasib...",None



Final Answer: 580 (votes=4, verified=3)

Answer: 580 | Ground Truth: 580 | ✅
📊 Running Accuracy: 41/50 (82.0%)
------

